In [2]:
!pip install wandb decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 90.3 MB/s eta 0:00:00:00:01:01


In [2]:
import os
import glob
import random
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
from torch.utils.data import Dataset, DataLoader

import numpy as np
import cv2
from decord import VideoReader, cpu
from tqdm import tqdm



from transformers import (
    VivitImageProcessor,
    VivitForVideoClassification,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import f1_score
import wandb

# ---------------------------
# Reproducibility
# ---------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@dataclass
class CFG:
    # Adjust to the folder that directly contains train/ and val/
    # Example structure:
    # /kaggle/input/rwf-2000/RWF-2000/train/Fight/...
    data_root: str = "/kaggle/input/rwf-2000/RWF-2000"

    # ViViT model from Hugging Face
    model_name: str = "google/vivit-b-16x2-kinetics400"

    num_frames: int = 16           # clip length
    frame_sample_rate: int = 4     # temporal stride
    image_size: int = 224

    # ViViT-B is heavy -> small batch size recommended
    train_batch_size: int = 2
    val_batch_size: int = 2
    num_workers: int = 2

    num_epochs: int = 10
    learning_rate: float = 3e-5
    weight_decay: float = 1e-4
    warmup_ratio: float = 0.1

    mixed_precision: bool = True   # AMP

    output_dir: str = "./checkpoints_vivit_rwf2000"
    project_name: str = "rwf2000-video-violence-vivit"
    run_name: str = "vivit-b16x2-rwf2000"

    seed: int = 42

set_seed(CFG.seed)

os.makedirs(CFG.output_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
num_gpus = torch.cuda.device_count() if device == "cuda" else 0
print(f"Using device: {device} | GPUs: {num_gpus}")

Using device: cuda | GPUs: 2


In [ ]:

WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
use_wandb = WANDB_API_KEY != ""

if use_wandb:
    wandb.login(key=WANDB_API_KEY)
    cfg_dict = {field: getattr(CFG, field) for field in CFG.__annotations__.keys()}
    wandb_run = wandb.init(
        project=CFG.project_name,
        name=CFG.run_name,
        config=cfg_dict,
    )
    print("W&B logging enabled.")
else:
    wandb_run = None
    print("W&B logging disabled (no WANDB_API_KEY).")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: remal11151 (remal11151-innopolis-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B logging enabled.


In [4]:
# ============================================================
# 3. List videos & labels (RWF-2000)
# ============================================================

label2id = {"NonFight": 0, "Fight": 1}
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)

print("Label mapping:", label2id)

def get_video_paths(split: str):
    assert split in ["train", "val"]
    base_dir = os.path.join(CFG.data_root, split)

    video_paths, labels = [], []

    for class_name in ["Fight", "NonFight"]:
        class_dir = os.path.join(base_dir, class_name)
        class_label = label2id[class_name]

        exts = ("*.avi", "*.mp4", "*.mkv", "*.mov")
        for ext in exts:
            pattern = os.path.join(class_dir, ext)
            for p in glob.glob(pattern):
                video_paths.append(p)
                labels.append(class_label)

    print(f"[{split}] Found {len(video_paths)} videos")
    return video_paths, labels

train_paths, train_labels = get_video_paths("train")
val_paths, val_labels = get_video_paths("val")

assert len(train_paths) > 0, "No training videos found. Check CFG.data_root."
assert len(val_paths) > 0, "No validation videos found. Check CFG.data_root."


Label mapping: {'NonFight': 0, 'Fight': 1}
[train] Found 1600 videos
[val] Found 400 videos


In [5]:
# ============================================================
# 4. Frame sampling + Dataset
# ============================================================

def sample_frame_indices_train(clip_len, frame_sample_rate, total_frames):
    """
    Random temporal window for training.
    """
    converted_len = clip_len * frame_sample_rate
    if total_frames <= converted_len:
        return np.linspace(0, total_frames - 1, clip_len).astype(np.int64)
    end_idx = np.random.randint(converted_len, total_frames)
    start_idx = end_idx - converted_len
    indices = np.linspace(start_idx, end_idx - 1, num=clip_len).astype(np.int64)
    return np.clip(indices, 0, total_frames - 1)

def sample_frame_indices_val(clip_len, frame_sample_rate, total_frames):
    """
    Deterministic center temporal window for validation.
    Helps reduce metric noise.
    """
    converted_len = clip_len * frame_sample_rate
    if total_frames <= converted_len:
        return np.linspace(0, total_frames - 1, clip_len).astype(np.int64)
    center = total_frames // 2
    start_idx = max(0, center - converted_len // 2)
    end_idx = start_idx + converted_len
    if end_idx > total_frames:
        end_idx = total_frames
        start_idx = end_idx - converted_len
    indices = np.linspace(start_idx, end_idx - 1, num=clip_len).astype(np.int64)
    return np.clip(indices, 0, total_frames - 1)

class RWF2000ViViTDataset(Dataset):
    def __init__(self, video_paths, labels, image_processor, is_train=True):
        self.video_paths = video_paths
        self.labels = labels
        self.image_processor = image_processor
        self.is_train = is_train

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        path = self.video_paths[idx]
        label = self.labels[idx]

        vr = VideoReader(path, num_threads=1, ctx=cpu(0))
        total_frames = len(vr)

        if self.is_train:
            indices = sample_frame_indices_train(
                CFG.num_frames, CFG.frame_sample_rate, total_frames
            )
        else:
            indices = sample_frame_indices_val(
                CFG.num_frames, CFG.frame_sample_rate, total_frames
            )

        buffer = vr.get_batch(indices).asnumpy()       # (T, H, W, C), uint8
        frames_list = [buffer[i] for i in range(buffer.shape[0])]

        # VivitImageProcessor handles resize, crop, normalize, etc.
        processed = self.image_processor(
            frames_list,
            return_tensors="pt",
        )
        # (1, T, C, H, W)
        pixel_values = processed["pixel_values"].squeeze(0)

        return {
            "pixel_values": pixel_values,               # (T, C, H, W)
            "labels": torch.tensor(label, dtype=torch.long),
            "path": path,
        }

def collate_fn(batch):
    pixel_values = torch.stack([b["pixel_values"] for b in batch])  # (B, T, C, H, W)
    labels = torch.stack([b["labels"] for b in batch])              # (B,)
    return {"pixel_values": pixel_values, "labels": labels}


In [9]:
# ============================================================
# 5. ViViT model & dataloaders (multi-GPU aware)
# ============================================================

image_processor = VivitImageProcessor.from_pretrained(CFG.model_name)

model = VivitForVideoClassification.from_pretrained(
    CFG.model_name,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,   # replace original head with 2-class head
)

model.to(device)

# ---- NEW: make sure we use the same num_frames as the model expects ----
if hasattr(model.config, "num_frames") and model.config.num_frames is not None:
    CFG.num_frames = model.config.num_frames
print("Model expects num_frames =", CFG.num_frames)
# ------------------------------------------------------------------------


# ---------- NEW: use multiple GPUs if available ----------
if device == "cuda" and torch.cuda.device_count() > 1:
    print(f"Using DataParallel on {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)
# ---------------------------------------------------------

train_dataset = RWF2000ViViTDataset(train_paths, train_labels, image_processor, is_train=True)
val_dataset   = RWF2000ViViTDataset(val_paths,   val_labels,   image_processor, is_train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.train_batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    collate_fn=collate_fn,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.val_batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_fn,
    pin_memory=True,
)

print("train batches:", len(train_loader), "val batches:", len(val_loader))


Some weights of VivitForVideoClassification were not initialized from the model checkpoint at google/vivit-b-16x2-kinetics400 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([400, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([400]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model expects num_frames = 32
Using DataParallel on 2 GPUs
train batches: 800 val batches: 200


In [10]:
# ============================================================
# 6. Optimizer, scheduler, AMP scaler
# ============================================================

total_train_steps = len(train_loader) * CFG.num_epochs
warmup_steps = int(CFG.warmup_ratio * total_train_steps)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
)

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_train_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and CFG.mixed_precision))


/tmp/ipykernel_102/1301197855.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and CFG.mixed_precision))


In [13]:
# ============================================================
# 7. Train & validate (with checkpoints)
# ============================================================

best_val_f1 = 0.0
global_step = 0

def train_one_epoch(epoch: int):
    global global_step
    model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG.num_epochs} [train]")
    for step, batch in enumerate(pbar):
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device == "cuda" and CFG.mixed_precision)):
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss
            # DataParallel returns one loss per GPU -> reduce to scalar
            if loss.ndim > 0:
                loss = loss.mean()
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()

        running_loss += loss.item()
        global_step += 1

        if step % 10 == 0:
            avg_loss = running_loss / (step + 1)
            current_lr = lr_scheduler.get_last_lr()[0]
            pbar.set_postfix({"loss": f"{avg_loss:.4f}", "lr": f"{current_lr:.2e}"})

            if use_wandb:
                wandb.log(
                    {
                        "train/loss": avg_loss,
                        "train/lr": current_lr,
                        "train/step": global_step,
                        "epoch": epoch + 1,
                    },
                    step=global_step,
                )

def validate(epoch: int):
    global best_val_f1
    model.eval()

    val_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{CFG.num_epochs} [val]")
        for batch in pbar:
            pixel_values = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=(device == "cuda" and CFG.mixed_precision)):
                outputs = model(pixel_values=pixel_values, labels=labels)
                loss = outputs.loss
                if loss.ndim > 0:
                    loss = loss.mean()
                logits = outputs.logits
            
            val_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_val_loss = val_loss / len(val_loader)
    all_preds_np = np.array(all_preds)
    all_labels_np = np.array(all_labels)

    accuracy = (all_preds_np == all_labels_np).mean().item()
    f1 = f1_score(all_labels_np, all_preds_np, average="macro")

    print(f"\nVal: loss={avg_val_loss:.4f} acc={accuracy:.4f} f1={f1:.4f}")

    if use_wandb:
        wandb.log(
            {
                "val/loss": avg_val_loss,
                "val/accuracy": accuracy,
                "val/f1": f1,
                "epoch": epoch + 1,
            },
            step=global_step,
        )

    # Helper: unwrap model if DataParallel
    model_to_save = model.module if isinstance(model, torch.nn.DataParallel) else model

    # Save best F1 checkpoint
    if f1 > best_val_f1:
        best_val_f1 = f1
        best_dir = os.path.join(CFG.output_dir, "best_model")
        os.makedirs(best_dir, exist_ok=True)
        print(f"New best F1 {best_val_f1:.4f} → saving to {best_dir}")
        model_to_save.save_pretrained(best_dir)
        image_processor.save_pretrained(best_dir)

    # Always save last checkpoint
    last_dir = os.path.join(CFG.output_dir, "last")
    os.makedirs(last_dir, exist_ok=True)
    model_to_save.save_pretrained(last_dir)
    image_processor.save_pretrained(last_dir)

    return avg_val_loss, accuracy, f1

for epoch in range(CFG.num_epochs):
    train_one_epoch(epoch)
    validate(epoch)

if wandb_run is not None:
    wandb.finish()

print("Training finished. Best F1:", best_val_f1)


Epoch 1/10 [train]:   2%|▏         | 14/800 [00:08<07:29,  1.75it/s, loss=0.2658, lr=3.00e-05]wandb: WARNING Tried to log to step 1 that is less than the current step 791. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 11 that is less than the current step 791. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
Epoch 1/10 [train]:   4%|▍         | 31/800 [00:18<07:27,  1.72it/s, loss=0.3197, lr=3.00e-05]wandb: WARNING Tried to log to step 21 that is less than the current step 791. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 31 that is less than the current step 791. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric


Val: loss=0.3251 acc=0.8575 f1=0.8574
New best F1 0.8574 → saving to ./checkpoints_vivit_rwf2000/best_model


Epoch 2/10 [train]:   0%|          | 0/800 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Epoch 2/10 [val]: 100%|██████████| 200/200 [01:25<00:00,  2.34it/s]



Val: loss=0.4180 acc=0.8400 f1=0.8384


Epoch 3/10 [train]:   0%|          | 0/800 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Epoch 3/10 [val]: 100%|██████████| 200/200 [01:28<00:00,  2.26it/s]


...


Val: loss=0.6185 acc=0.8650 f1=0.8648
New best F1 0.8648 → saving to ./checkpoints_vivit_rwf2000/best_model


Epoch 7/10 [train]:   0%|          | 0/800 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Epoch 7/10 [val]: 100%|██████████| 200/200 [01:23<00:00,  2.38it/s]



Val: loss=0.6554 acc=0.8650 f1=0.8648


Epoch 8/10 [train]:   0%|          | 0/800 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Epoch 8/10 [train]:  62%|██████▏   | 493/800 [03:52<02:07,  2.41it/s, loss=0.0023, lr=1.72e-06]Exception ignored in: <function _xla_gc_callback at 0x7e6e197bc180>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 
Epoch 8/10 [train]:  62%|██████▏   | 495/800 [03:58<02:27,  2.07it/s, loss=0.0023, lr=1.72e-06]


RuntimeError: DataLoader worker (pid(s) 1015115, 1015116) exited unexpectedly

In [14]:
import shutil
from IPython.display import FileLink

folder_path = "/kaggle/working/checkpoints_vivit_rwf2000"
zip_path = "/kaggle/working/checkpoints_vivit_rwf2000"  # without .zip

# create zip
shutil.make_archive(zip_path, 'zip', folder_path)

# show download link
FileLink('checkpoints_vivit_rwf2000.zip')

/kaggle/working/checkpoints_vivit_rwf2000.zip